
SLIMA Optimized Training Pipeline - 6×H100 GPUs
================================================

Complete training, validation, and test pipeline for histologic pattern classification.
Combines your existing Accelerate setup with all optimizations for 90% target accuracy.

Key Improvements:
1. Train/Val/Test split (60/20/20)
2. Strong histology-specific augmentation
3. Cosine LR schedule with warmup
4. Label smoothing
5. Larger image size (384)
6. Extended training (200 epochs)
7. Test-Time Augmentation for final evaluation

Usage:
    Launch from notebook: launch_from_notebook(num_processes=6)
"""


In [1]:
# CELL 0: MUST BE FIRST CELL AFTER KERNEL RESTART
import torch.multiprocessing as mp
try:
    mp.set_start_method('spawn', force=True)
except RuntimeError:
    pass  # Already set

In [2]:
# Prevent CUDA init at import time
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0,1,2,3,4,5' 

In [3]:

import os
import re
import sys
import json
import time
import random
import math
from pathlib import Path
from collections import Counter
from typing import Dict, List, Tuple, Optional
from dataclasses import dataclass

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from torchvision import models, transforms
from torchvision.transforms import functional as TF
from torchvision.transforms import InterpolationMode

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report,
    precision_score, recall_score, confusion_matrix
)

try:
    from scipy.io import loadmat
except Exception:
    loadmat = None
try:
    import h5py
except Exception:
    h5py = None

from accelerate import Accelerator, notebook_launcher

# Optional: albumentations for stronger augmentation
try:
    import albumentations as A
    from albumentations.pytorch import ToTensorV2
    HAS_ALBUMENTATIONS = True
except ImportError:
    HAS_ALBUMENTATIONS = False
    print("Note: albumentations not installed, using torchvision transforms")



In [4]:

# ==============================================================================
# CONFIGURATION
# ==============================================================================

@dataclass
class Config:
    """Optimized configuration for 90% target accuracy."""
    
    # Data paths
    ROOT_DIR: str = "/home/rapids/notebooks/slima/Zenodo_Anorak_original"
    IMAGE_DIR: str = None  # Set in __post_init__
    MASK_DIR: str = None   # Set in __post_init__
    XLS_PATH: str = "/home/rapids/notebooks/slima/overlay_index ver 9 nov 2025.xlsx"
    OUT_DIR: str = "/home/rapids/notebooks/slima/outputs/optimized_90_target"
    
    # Classes
    INCLUDE_PATTERNS: str = "lepidic,acinar,papillary,micropapillary,solid,mucinous"
    
    # Model
    ARCH: str = "resnet101"
    EMBED_DIM: int = 512
    
    # Training - EXTENDED
    EPOCHS: int = 200           # Increased from 120
    BATCH_SIZE: int = 24        # Per GPU (adjust if OOM)
    LR: float = 1e-4            # Lower initial LR
    LR_BACKBONE: float = 1e-5   # Even lower for pretrained backbone
    WEIGHT_DECAY: float = 1e-4
    WARMUP_EPOCHS: int = 10     # Warmup period
    
    # Image
    IMG_SIZE: int = 384
    
    # FuzzyArcLoss - Optimized from Optuna
    S_SCALE: float = 21.97
    M_MARGIN: float = 0.23
    TAU: float = 0.65
    
    # ROI settings
    USE_MASK_AS_CHANNEL: bool = True
    USE_ROI_CROP: bool = True
    ROI_PADDING: int = 24
    MASK_THRESH: float = 0.5
    
    # Data splits
    VAL_SIZE: float = 0.15      # 15% for validation
    TEST_SIZE: float = 0.15     # 15% for test (held out)
    SEED: int = 42
    
    # Distributed

    NUM_WORKERS = 0  # start here to confirm stability
    PREFETCH_FACTOR = None  # only relevant if num_workers > 0
    PERSISTENT_WORKERS = False
    PIN_MEMORY = False

    NUM_PROCESSES: int = 6
    #NUM_WORKERS: int = 6
    MIXED_PRECISION: str = "no"
    GRAD_ACCUM_STEPS: int = 1
    
    # Regularization
    LABEL_SMOOTHING: float = 0.1
    DROPOUT: float = 0.1
    
    # Test-Time Augmentation
    USE_TTA: bool = True
    TTA_AUGMENTATIONS: int = 5
    
    def __post_init__(self):
        if self.IMAGE_DIR is None:
            self.IMAGE_DIR = f"{self.ROOT_DIR}/image"
        if self.MASK_DIR is None:
            self.MASK_DIR = f"{self.ROOT_DIR}/mask"


# Global config instance
config = Config()

# # Enable TF32 for H100
# torch.backends.cuda.matmul.allow_tf32 = True
# try:
#     torch.set_float32_matmul_precision("high")
# except Exception:
#     pass



In [5]:

# ==============================================================================
# UTILITIES
# ==============================================================================

IMG_EXTS = {".png", ".jpg", ".jpeg", ".tif", ".tiff"}
MAT_EXCLUDE = {"__header__", "__version__", "__globals__"}
MAT_PRIOR = ["mask", "Mask", "BW", "bw", "label", "Label", "roi", "ROI", "seg", "Seg"]


def ensure_dir(p: str):
    os.makedirs(p, exist_ok=True)


# def set_seed(seed: int = 42):
#     random.seed(seed)
#     np.random.seed(seed)
#     torch.manual_seed(seed)
#     torch.cuda.manual_seed_all(seed)

def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    # DO NOT call torch.cuda.manual_seed_all() here


def list_images(d: str) -> List[Path]:
    out = []
    base = Path(d)
    for p in base.rglob("*"):
        if p.is_file() and p.suffix.lower() in IMG_EXTS:
            out.append(p)
    return sorted(out)


def list_mats(d: str) -> List[Path]:
    return sorted([p for p in Path(d).rglob("*.mat") if p.is_file()])


def normalize_key(stem: str) -> str:
    s = stem.lower()
    s = re.sub(r"([_-])(mask|seg|roi|label|overlay)([_-]?\d+)?$", "", s)
    s = re.sub(r"[ \t\-_]+", "_", s).strip("_")
    return s


def build_mat_index(mask_dir: str) -> Dict[str, str]:
    idx = {}
    for p in list_mats(mask_dir):
        stem = p.stem
        for key in (stem, stem.lower(), normalize_key(stem)):
            if key not in idx or p.stat().st_size > Path(idx[key]).stat().st_size:
                idx[key] = str(p)
    return idx


def pair_images_with_masks(image_dir: str, mask_dir: str) -> pd.DataFrame:
    imgs = list_images(image_dir)
    mat_index = build_mat_index(mask_dir)
    
    rows, unmatched = [], []
    for ip in imgs:
        stem = ip.stem
        key_norm = normalize_key(stem)
        mp = mat_index.get(stem) or mat_index.get(stem.lower()) or mat_index.get(key_norm)
        if mp:
            rows.append({
                "image_path": str(ip),
                "mask_path": mp,
                "base": stem,
                "base_norm": key_norm
            })
        else:
            unmatched.append(ip.name)
    
    print(f"[PAIRING] matched={len(rows)} unmatched={len(unmatched)}")
    return pd.DataFrame(rows)


def read_labels_from_xls(xls_path: str) -> Tuple[Dict[str, str], str]:
    df = pd.read_excel(xls_path)
    label_candidates = ["pattern", "label", "class", "type", "histologic_pattern"]
    file_candidates = ["tile_id", "overlay_file", "image_file", "file", "filename"]
    
    label_col = next((c for c in label_candidates if c in df.columns), None)
    if label_col is None:
        raise ValueError(f"No label column found in {xls_path}")
    
    file_col = next((c for c in file_candidates if c in df.columns), None)
    if file_col is None:
        raise ValueError(f"No file column found in {xls_path}")
    
    m = {}
    for _, r in df.iterrows():
        f = str(r[file_col])
        stem = Path(f).stem
        lbl = str(r[label_col]).strip()
        m[stem] = lbl
        m[normalize_key(stem)] = lbl
        m[stem.lower()] = lbl
    
    return m, label_col


def _load_mat_any(path: str) -> np.ndarray:
    if loadmat is not None:
        try:
            d = loadmat(path)
            for k in MAT_PRIOR:
                if k in d and isinstance(d[k], np.ndarray) and d[k].ndim >= 2:
                    return d[k]
            best, sz = None, -1
            for k, v in d.items():
                if k in MAT_EXCLUDE:
                    continue
                if isinstance(v, np.ndarray) and v.ndim >= 2:
                    s = np.prod(v.shape[:2])
                    if s > sz:
                        best, sz = v, s
            if best is not None:
                return best
        except Exception:
            pass
    
    if h5py is not None:
        try:
            with h5py.File(path, "r") as f:
                for k in MAT_PRIOR:
                    if k in f and f[k].ndim >= 2:
                        return np.array(f[k])
                for k in f.keys():
                    if k not in MAT_EXCLUDE and f[k].ndim >= 2:
                        return np.array(f[k])
        except Exception:
            pass
    
    return np.ones((224, 224), dtype=np.uint8)


def load_mask_from_mat(path: str) -> Image.Image:
    arr = _load_mat_any(path)
    if arr.ndim == 3 and arr.shape[0] in (1, 2, 3, 4):
        arr = arr[0]
    elif arr.ndim == 3 and arr.shape[-1] in (1, 2, 3, 4):
        arr = arr[..., 0]
    if arr.ndim > 2:
        arr = arr[..., 0] if arr.shape[-1] <= 4 else np.argmax(arr, axis=-1)
    mask = (arr.astype(np.float32) > 0).astype(np.uint8) * 255
    return Image.fromarray(mask, mode="L")



In [6]:

# ==============================================================================
# DATASET WITH STRONG AUGMENTATION
# ==============================================================================

class HistMaskDataset(Dataset):
    """Dataset with strong histology-specific augmentation."""
    
    def __init__(
        self,
        rows: pd.DataFrame,
        label_col: str,
        label2id: Dict[str, int],
        img_size: int = 384,
        aug: bool = True,
        use_mask_as_channel: bool = True,
        use_roi_crop: bool = True,
        roi_padding: int = 24,
        mask_thresh: float = 0.5,
        strong_aug: bool = True,
    ):
        self.rows = rows.reset_index(drop=True)
        self.label_col = label_col
        self.label2id = label2id
        self.img_size = img_size
        self.aug = aug
        self.use_mask_as_channel = use_mask_as_channel
        self.use_roi_crop = use_roi_crop
        self.roi_padding = roi_padding
        self.mask_thresh = mask_thresh
        self.strong_aug = strong_aug and aug
        
        # Color jitter for basic augmentation
        self.cj = transforms.ColorJitter(0.3, 0.3, 0.3, 0.1)
    
    def __len__(self):
        return len(self.rows)
    
    def _roi_crop(self, img: Image.Image, msk: Image.Image):
        m = np.array(msk) > 0
        if not m.any():
            return img, msk
        ys, xs = np.where(m)
        y0 = max(0, ys.min() - self.roi_padding)
        y1 = min(m.shape[0], ys.max() + 1 + self.roi_padding)
        x0 = max(0, xs.min() - self.roi_padding)
        x1 = min(m.shape[1], xs.max() + 1 + self.roi_padding)
        return img.crop((x0, y0, x1, y1)), msk.crop((x0, y0, x1, y1))
    
    def _joint_transform(self, img: Image.Image, msk: Image.Image):
        """Apply synchronized transforms to image and mask."""
        if self.aug:
            # Random resized crop
            i, j, h, w = transforms.RandomResizedCrop.get_params(
                img, scale=(0.6, 1.0), ratio=(0.9, 1.1)
            )
            img = TF.resized_crop(img, i, j, h, w, 
                                  size=[self.img_size, self.img_size],
                                  interpolation=InterpolationMode.BILINEAR)
            msk = TF.resized_crop(msk, i, j, h, w,
                                  size=[self.img_size, self.img_size],
                                  interpolation=InterpolationMode.NEAREST)
            
            # Flips
            if random.random() < 0.5:
                img = TF.hflip(img)
                msk = TF.hflip(msk)
            if random.random() < 0.5:
                img = TF.vflip(img)
                msk = TF.vflip(msk)
            
            # Rotation
            if random.random() < 0.5:
                angle = random.uniform(-30, 30)
                img = TF.rotate(img, angle, interpolation=InterpolationMode.BILINEAR)
                msk = TF.rotate(msk, angle, interpolation=InterpolationMode.NEAREST)
            
            # Strong augmentation (histology-specific)
            if self.strong_aug:
                # Gaussian blur
                if random.random() < 0.3:
                    img = TF.gaussian_blur(img, kernel_size=random.choice([3, 5]))
                
                # Adjust sharpness
                if random.random() < 0.2:
                    img = TF.adjust_sharpness(img, sharpness_factor=random.uniform(1.5, 2.5))
                
                # Posterize (simulates stain variation)
                if random.random() < 0.2:
                    img = TF.posterize(img, bits=random.choice([4, 5, 6]))
                
                # Solarize
                if random.random() < 0.1:
                    img = TF.solarize(img, threshold=random.randint(128, 200))
                
                # Equalize
                if random.random() < 0.1:
                    img = TF.equalize(img)
        else:
            img = img.resize((self.img_size, self.img_size), resample=Image.BILINEAR)
            msk = msk.resize((self.img_size, self.img_size), resample=Image.NEAREST)
        
        return img, msk
    
    def __getitem__(self, idx: int):
        r = self.rows.iloc[idx]
        img = Image.open(r["image_path"]).convert("RGB")
        msk = load_mask_from_mat(r["mask_path"])
        
        # ROI crop
        if self.use_roi_crop:
            img, msk = self._roi_crop(img, msk)
        
        # Joint transforms
        img, msk = self._joint_transform(img, msk)
        
        # Color augmentation (only on image)
        if self.aug:
            img = self.cj(img)
        
        # To tensor and normalize
        img_t = TF.to_tensor(img)
        img_t = TF.normalize(img_t, mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        msk_t = (TF.to_tensor(msk) > self.mask_thresh).float()
        
        # Combine image and mask
        x = torch.cat([img_t, msk_t], dim=0) if self.use_mask_as_channel else img_t
        y = self.label2id[str(r[self.label_col])]
        
        return x, y, r["image_path"]



In [7]:

# ==============================================================================
# MODEL COMPONENTS
# ==============================================================================

class LabelSmoothingCrossEntropy(nn.Module):
    """Cross entropy with label smoothing."""
    
    def __init__(self, smoothing: float = 0.1, weight: torch.Tensor = None):
        super().__init__()
        self.smoothing = smoothing
        self.register_buffer("weight", weight)
    
    def forward(self, pred: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        n_classes = pred.size(-1)
        log_preds = F.log_softmax(pred, dim=-1)
        
        with torch.no_grad():
            true_dist = torch.zeros_like(pred)
            true_dist.fill_(self.smoothing / (n_classes - 1))
            true_dist.scatter_(1, target.unsqueeze(1), 1.0 - self.smoothing)
        
        loss = (-true_dist * log_preds).sum(dim=-1)
        
        if self.weight is not None:
            loss = loss * self.weight[target]
        
        return loss.mean()


class FuzzyArcMarginProduct(nn.Module):
    """Fuzzy Arc Margin Product for angular margin loss."""
    
    def __init__(self, in_features: int, out_features: int, 
                 s: float = 30.0, m: float = 0.50, tau: float = 0.10):
        super().__init__()
        self.s = float(s)
        self.m = float(m)
        self.tau = float(tau)
        self.weight = nn.Parameter(torch.empty(out_features, in_features))
        nn.init.xavier_uniform_(self.weight)
    
    def forward(self, features: torch.Tensor, labels: torch.Tensor) -> torch.Tensor:
        x = F.normalize(features, dim=1)
        W = F.normalize(self.weight, dim=1)
        cos = (x @ W.t()).clamp(-1, 1)
        
        idx = torch.arange(x.size(0), device=x.device)
        cos_y = cos[idx, labels]
        
        # Fuzzy membership
        mu = torch.where(
            torch.abs(cos_y) >= self.tau,
            torch.abs(cos_y),
            torch.ones_like(cos_y)
        )
        m_eff = self.m * mu
        
        # Angular margin
        cos_m = torch.cos(m_eff)
        sin_m = torch.sin(m_eff)
        sin_t = torch.sqrt((1 - cos_y ** 2).clamp(0, 1))
        cos_theta_m = cos_y * cos_m - sin_t * sin_m
        
        logits = cos * self.s
        logits[idx, labels] = cos_theta_m * self.s
        
        return logits


class FuzzyArcLoss(nn.Module):
    """Complete FuzzyArcLoss module."""
    
    def __init__(self, in_features: int, out_features: int,
                 s: float = 30.0, m: float = 0.50, tau: float = 0.10,
                 label_smoothing: float = 0.1, ce_weight: torch.Tensor = None):
        super().__init__()
        self.head = FuzzyArcMarginProduct(in_features, out_features, s=s, m=m, tau=tau)
        self.ce = LabelSmoothingCrossEntropy(smoothing=label_smoothing, weight=ce_weight)
    
    def forward(self, feats: torch.Tensor, labels: torch.Tensor):
        logits = self.head(feats, labels)
        loss = self.ce(logits, labels)
        return loss, logits


def build_backbone(arch: str = "resnet101", pretrained: bool = True,
                   embed_dim: int = 512, in_channels: int = 3,
                   dropout: float = 0.1) -> Tuple[nn.Module, int]:
    """Build backbone with embedding head."""
    
    if arch == "resnet50":
        m = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2 if pretrained else None)
    elif arch == "resnet101":
        m = models.resnet101(weights=models.ResNet101_Weights.IMAGENET1K_V2 if pretrained else None)
    else:
        raise ValueError(f"Unknown architecture: {arch}")
    
    # Modify first conv for 4-channel input
    if in_channels != 3:
        old = m.conv1
        m.conv1 = nn.Conv2d(
            in_channels, old.out_channels,
            kernel_size=old.kernel_size,
            stride=old.stride,
            padding=old.padding,
            bias=False
        )
        with torch.no_grad():
            m.conv1.weight[:, :3] = old.weight
            if in_channels > 3:
                mean_rgb = old.weight.mean(dim=1, keepdim=True)
                m.conv1.weight[:, 3:in_channels] = mean_rgb.repeat(1, in_channels - 3, 1, 1)
    
    in_dim = m.fc.in_features
    m.fc = nn.Identity()
    
    # Embedding head with dropout
    head = nn.Sequential(
        nn.BatchNorm1d(in_dim),
        nn.Dropout(dropout),
        nn.Linear(in_dim, embed_dim, bias=False),
        nn.BatchNorm1d(embed_dim),
    )
    
    net = nn.Sequential(m, head)
    return net, embed_dim


# ==============================================================================
# LEARNING RATE SCHEDULE
# ==============================================================================

def get_cosine_schedule_with_warmup(optimizer, warmup_epochs: int, total_epochs: int,
                                    min_lr_ratio: float = 0.01):
    """Cosine annealing with linear warmup."""
    
    def lr_lambda(epoch):
        if epoch < warmup_epochs:
            return float(epoch) / float(max(1, warmup_epochs))
        progress = float(epoch - warmup_epochs) / float(max(1, total_epochs - warmup_epochs))
        return max(min_lr_ratio, 0.5 * (1.0 + math.cos(math.pi * progress)))
    
    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)


# ==============================================================================
# TEST-TIME AUGMENTATION
# ==============================================================================

def apply_tta(model, loss_head, x, device, n_augmentations=5):
    """Apply test-time augmentation and average predictions."""
    
    all_probs = []
    
    # Original
    with torch.no_grad():
        feats = model(x)
        # For inference, we need logits without margin
        W = F.normalize(loss_head.head.weight, dim=1)
        feats_norm = F.normalize(feats, dim=1)
        logits = (feats_norm @ W.t()) * loss_head.head.s
        probs = F.softmax(logits, dim=-1)
        all_probs.append(probs)
    
    # Augmented versions
    augmentations = [
        lambda t: torch.flip(t, dims=[-1]),      # Horizontal flip
        lambda t: torch.flip(t, dims=[-2]),      # Vertical flip
        lambda t: torch.flip(t, dims=[-1, -2]),  # Both flips
        lambda t: torch.rot90(t, k=1, dims=[-2, -1]),  # 90° rotation
    ]
    
    for aug_fn in augmentations[:n_augmentations - 1]:
        with torch.no_grad():
            x_aug = aug_fn(x)
            feats = model(x_aug)
            feats_norm = F.normalize(feats, dim=1)
            logits = (feats_norm @ W.t()) * loss_head.head.s
            probs = F.softmax(logits, dim=-1)
            all_probs.append(probs)
    
    # Average
    avg_probs = torch.stack(all_probs).mean(dim=0)
    return avg_probs



In [8]:

# ==============================================================================
# MAIN TRAINING FUNCTION
# ==============================================================================

def training_function():
    """Main training function with train/val/test splits."""
    
    accelerator = Accelerator(
        mixed_precision=config.MIXED_PRECISION,
        gradient_accumulation_steps=config.GRAD_ACCUM_STEPS
    )

    set_seed(config.SEED)
    if accelerator.device.type == "cuda":
        torch.cuda.manual_seed_all(config.SEED)
    
    if accelerator.is_main_process:
        ensure_dir(config.OUT_DIR)
    
    accelerator.print(f"Running on {accelerator.state.num_processes} GPUs")
    accelerator.print(f"Mixed precision: {accelerator.mixed_precision}")
    
    #set_seed(config.SEED)
    
    # ==== 1. Load and pair data ====
    accelerator.print(f"\n[1/7] Loading data...")
    df = pair_images_with_masks(config.IMAGE_DIR, config.MASK_DIR)
    if df.empty:
        raise RuntimeError("No image-mask pairs found!")
    
    # ==== 2. Map labels ====
    label_map, label_col = read_labels_from_xls(config.XLS_PATH)
    df[label_col] = df["base"].map(label_map)
    miss = df[label_col].isna()
    if miss.any():
        df.loc[miss, label_col] = df.loc[miss, "base_norm"].map(label_map)
    df = df[~df[label_col].isna()].copy()
    accelerator.print(f"[LABELS] after XLS join: {len(df)} rows")
    
    # ==== 3. Filter classes ====
    inc = {p.strip().lower() for p in config.INCLUDE_PATTERNS.split(",") if p.strip()}
    df[label_col] = df[label_col].astype(str)
    df = df[df[label_col].str.lower().isin(inc)].copy()
    accelerator.print(f"[FILTER] kept={len(df)} over classes={sorted(list(inc))}")
    
    if df.empty:
        raise RuntimeError("Dataset empty after filtering!")
    
    # ==== 4. Label mappings ====
    classes = sorted(df[label_col].unique().tolist())
    label2id = {c: i for i, c in enumerate(classes)}
    id2label = {v: k for k, v in label2id.items()}
    n_classes = len(classes)
    
    if accelerator.is_main_process:
        print(f"[CLASSES] {label2id}")
    
    # ==== 5. Train/Val/Test split ====
    accelerator.print(f"\n[2/7] Splitting data (train/val/test)...")
    
    # First split: separate test set
    train_val_df, test_df = train_test_split(
        df, test_size=config.TEST_SIZE,
        random_state=config.SEED, stratify=df[label_col]
    )
    
    # Second split: train and validation
    adjusted_val_size = config.VAL_SIZE / (1 - config.TEST_SIZE)
    train_df, val_df = train_test_split(
        train_val_df, test_size=adjusted_val_size,
        random_state=config.SEED, stratify=train_val_df[label_col]
    )
    
    accelerator.print(f"  Train: {len(train_df)} samples")
    accelerator.print(f"  Val:   {len(val_df)} samples")
    accelerator.print(f"  Test:  {len(test_df)} samples")
    
    # Class distribution
    if accelerator.is_main_process:
        print("\nClass distribution:")
        for split_name, split_df in [("Train", train_df), ("Val", val_df), ("Test", test_df)]:
            counts = Counter(split_df[label_col])
            print(f"  {split_name}: {dict(counts)}")
    
    # ==== 6. Compute class weights ====
    counts = Counter(train_df[label_col])
    ce_weights = torch.tensor(
        [1.0 / counts[id2label[i]] for i in range(n_classes)],
        dtype=torch.float32
    )
    ce_weights = ce_weights / ce_weights.sum() * n_classes  # Normalize
    
    # ==== 7. Create datasets ====
    accelerator.print(f"\n[3/7] Creating datasets...")
    in_ch = 4 if config.USE_MASK_AS_CHANNEL else 3
    
    train_ds = HistMaskDataset(
        train_df, label_col, label2id, config.IMG_SIZE,
        aug=True, use_mask_as_channel=config.USE_MASK_AS_CHANNEL,
        use_roi_crop=config.USE_ROI_CROP, roi_padding=config.ROI_PADDING,
        mask_thresh=config.MASK_THRESH, strong_aug=True
    )
    
    val_ds = HistMaskDataset(
        val_df, label_col, label2id, config.IMG_SIZE,
        aug=False, use_mask_as_channel=config.USE_MASK_AS_CHANNEL,
        use_roi_crop=config.USE_ROI_CROP, roi_padding=config.ROI_PADDING,
        mask_thresh=config.MASK_THRESH, strong_aug=False
    )
    
    test_ds = HistMaskDataset(
        test_df, label_col, label2id, config.IMG_SIZE,
        aug=False, use_mask_as_channel=config.USE_MASK_AS_CHANNEL,
        use_roi_crop=config.USE_ROI_CROP, roi_padding=config.ROI_PADDING,
        mask_thresh=config.MASK_THRESH, strong_aug=False
    )
    
    # train_ld = DataLoader(
    #     train_ds, batch_size=config.BATCH_SIZE, shuffle=True,
    #     num_workers=config.NUM_WORKERS, pin_memory=True, drop_last=True
    # )

    train_ld = DataLoader(
        train_ds,
        batch_size=config.BATCH_SIZE,
        shuffle=True,
        num_workers=config.NUM_WORKERS,     # 0 first
        pin_memory=False,                   # important
        persistent_workers=False,           # important
    )

    
    val_ld = DataLoader(
        val_ds, batch_size=config.BATCH_SIZE, shuffle=False,
        num_workers=config.NUM_WORKERS, pin_memory=True
    )
    test_ld = DataLoader(
        test_ds, batch_size=config.BATCH_SIZE, shuffle=False,
        num_workers=config.NUM_WORKERS, pin_memory=True
    )
    
    # ==== 8. Build model ====
    accelerator.print(f"\n[4/7] Building model...")
    model, _ = build_backbone(
        config.ARCH, pretrained=True, embed_dim=config.EMBED_DIM,
        in_channels=in_ch, dropout=config.DROPOUT
    )
    
    loss_head = FuzzyArcLoss(
        config.EMBED_DIM, n_classes,
        s=config.S_SCALE, m=config.M_MARGIN, tau=config.TAU,
        label_smoothing=config.LABEL_SMOOTHING,
        ce_weight=ce_weights.to(accelerator.device)
    )
    
    # ==== 9. Optimizer with differential LR ====
    # Backbone gets lower LR, head gets higher LR
    backbone_params = list(model[0].parameters())
    head_params = list(model[1].parameters()) + list(loss_head.parameters())
    
    optimizer = torch.optim.AdamW([
        {"params": backbone_params, "lr": config.LR_BACKBONE},
        {"params": head_params, "lr": config.LR},
    ], weight_decay=config.WEIGHT_DECAY)
    
    scheduler = get_cosine_schedule_with_warmup(
        optimizer, config.WARMUP_EPOCHS, config.EPOCHS
    )
    
    # ==== 10. Prepare for distributed ====
    model, loss_head, optimizer, train_ld, val_ld, test_ld = accelerator.prepare(
        model, loss_head, optimizer, train_ld, val_ld, test_ld
    )
    
    # ==== 11. Training loop ====
    accelerator.print(f"\n[5/7] Starting training for {config.EPOCHS} epochs...")
    
    best_metric = -1.0
    best_epoch = 0
    best_path = os.path.join(config.OUT_DIR, "best_model3.pth")
    history = {"train_loss": [], "val_loss": [], "val_acc": [], "val_f1": [], "lr": []}


    if os.path.exists(best_path):
        accelerator.print(f"\n[SKIP] Found existing model at {best_path}")
        accelerator.print("[SKIP] Skipping training, loading checkpoint for evaluation...")
        
        # Load checkpoint to get best_metric and best_epoch
        checkpoint = torch.load(best_path, map_location=accelerator.device, weights_only=False)
        best_metric = checkpoint.get('best_metric', 0.0)
        best_epoch = checkpoint.get('epoch', 0)

        # IMPORTANT: Load weights into model BEFORE evaluation
        accelerator.unwrap_model(model).load_state_dict(checkpoint["model_state"])
        accelerator.unwrap_model(loss_head).load_state_dict(checkpoint["head_state"])
        
        # Load history if available
        if 'history' in checkpoint:
            history = checkpoint['history']

        
        
        accelerator.print(f"[SKIP] Loaded model from epoch {best_epoch} with metric {best_metric:.4f}")
        
    else:
   
     
        for epoch in range(1, config.EPOCHS + 1):
            t0 = time.time()
            
            # ---- Training ----
            model.train()
            loss_head.train()
            train_loss = 0.0
            train_preds, train_labels = [], []
            
            for x, y, _ in train_ld:
                with accelerator.accumulate(model):
                    feats = model(x)
                    loss, logits = loss_head(feats, y)
                    accelerator.backward(loss)
                    optimizer.step()
                    optimizer.zero_grad(set_to_none=True)
                
                train_loss += loss.detach().float() * x.size(0)
                pred = torch.argmax(logits.detach(), dim=1)
                train_preds.append(accelerator.gather(pred).cpu())
                train_labels.append(accelerator.gather(y).cpu())
            
            train_preds = torch.cat(train_preds).numpy()
            train_labels = torch.cat(train_labels).numpy()
            train_acc = accuracy_score(train_labels, train_preds)
            train_f1 = f1_score(train_labels, train_preds, average="macro")
            train_loss = train_loss.item() / len(train_ds)
            
            # ---- Validation ----
            model.eval()
            loss_head.eval()
            val_loss = 0.0
            val_preds, val_labels = [], []
            
            with torch.no_grad():
                for x, y, _ in val_ld:
                    feats = model(x)
                    loss, logits = loss_head(feats, y)
                    val_loss += loss.detach().float() * x.size(0)
                    pred = torch.argmax(logits, dim=1)
                    val_preds.append(accelerator.gather(pred).cpu())
                    val_labels.append(accelerator.gather(y).cpu())
            
            val_preds = torch.cat(val_preds).numpy()
            val_labels = torch.cat(val_labels).numpy()
            val_acc = accuracy_score(val_labels, val_preds)
            val_f1 = f1_score(val_labels, val_preds, average="macro")
            val_precision = precision_score(val_labels, val_preds, average="macro", zero_division=0)
            val_recall = recall_score(val_labels, val_preds, average="macro", zero_division=0)
            val_loss = val_loss.item() / len(val_ds)
            
            # Metric for model selection
            val_metric = (val_precision + val_recall) / 2
            
            # Update scheduler
            scheduler.step()
            current_lr = optimizer.param_groups[0]["lr"]
            
            # Log
            history["train_loss"].append(train_loss)
            history["val_loss"].append(val_loss)
            history["val_acc"].append(val_acc)
            history["val_f1"].append(val_f1)
            history["lr"].append(current_lr)
            
            accelerator.print(
                f"[{epoch:03d}/{config.EPOCHS}] "
                f"tr_loss={train_loss:.4f} tr_acc={train_acc:.4f} | "
                f"val_loss={val_loss:.4f} val_acc={val_acc:.4f} val_f1={val_f1:.4f} | "
                f"P={val_precision:.4f} R={val_recall:.4f} avg={val_metric:.4f} | "
                f"lr={current_lr:.2e} | {time.time()-t0:.1f}s"
            )
            
            # Save best model
            if accelerator.is_main_process and val_metric > best_metric:
                best_metric = val_metric
                best_epoch = epoch
                state = {
                    "epoch": epoch,
                    "model_state": accelerator.unwrap_model(model).state_dict(),
                    "head_state": accelerator.unwrap_model(loss_head).state_dict(),
                    "optimizer_state": optimizer.state_dict(),
                    "scheduler_state": scheduler.state_dict(),
                    "label2id": label2id,
                    "id2label": id2label,
                    "best_metric": best_metric,
                    "config": config.__dict__,
                }
                torch.save(state, best_path)
                accelerator.print(f"  --> Saved best model (avg_P_R={best_metric:.4f})")
        
    accelerator.print(f"\n[6/7] Training complete! Best epoch: {best_epoch} with metric: {best_metric:.4f}")
    
    # ==== 12. Final Test Evaluation ====
    accelerator.print(f"\n[7/7] Evaluating on TEST set...")
    
    # Load best model
    if accelerator.is_main_process:
        #checkpoint = torch.load(best_path, map_location=accelerator.device)
        # CORRECT - add weights_only=False
        checkpoint = torch.load(best_path, map_location=accelerator.device, weights_only=False)

        accelerator.unwrap_model(model).load_state_dict(checkpoint["model_state"])
        accelerator.unwrap_model(loss_head).load_state_dict(checkpoint["head_state"])
    accelerator.wait_for_everyone()
    
    model.eval()
    loss_head.eval()
    test_preds, test_labels, test_probs = [], [], []
    
    with torch.no_grad():
        for x, y, _ in test_ld:
            if config.USE_TTA:
                # Test-time augmentation
                probs = apply_tta(
                    model, accelerator.unwrap_model(loss_head),
                    x, accelerator.device, n_augmentations=config.TTA_AUGMENTATIONS
                )
                pred = probs.argmax(dim=1)
            else:
                feats = model(x)
                _, logits = loss_head(feats, y)
                pred = torch.argmax(logits, dim=1)
                probs = F.softmax(logits, dim=-1)
            
            test_preds.append(accelerator.gather(pred).cpu())
            test_labels.append(accelerator.gather(y).cpu())
            test_probs.append(accelerator.gather(probs).cpu())
    
    test_preds = torch.cat(test_preds).numpy()
    test_labels = torch.cat(test_labels).numpy()
    
    # Final metrics
    test_acc = accuracy_score(test_labels, test_preds)
    test_f1 = f1_score(test_labels, test_preds, average="macro")
    test_precision = precision_score(test_labels, test_preds, average="macro", zero_division=0)
    test_recall = recall_score(test_labels, test_preds, average="macro", zero_division=0)
    
    accelerator.print("\n" + "=" * 80)
    accelerator.print("FINAL TEST RESULTS")
    accelerator.print("=" * 80)
    accelerator.print(f"  Accuracy:  {test_acc*100:.2f}%")
    accelerator.print(f"  F1-Score:  {test_f1*100:.2f}%")
    accelerator.print(f"  Precision: {test_precision*100:.2f}%")
    accelerator.print(f"  Recall:    {test_recall*100:.2f}%")
    accelerator.print(f"  Avg(P,R):  {(test_precision+test_recall)/2*100:.2f}%")
    
    if accelerator.is_main_process:
        print("\n=== Test Classification Report ===")
        print(classification_report(
            test_labels, test_preds,
            target_names=[id2label[i] for i in range(n_classes)]
        ))
        
        # Save final results
        results = {
            "test_accuracy": float(test_acc),
            "test_f1": float(test_f1),
            "test_precision": float(test_precision),
            "test_recall": float(test_recall),
            "best_epoch": best_epoch,
            "best_val_metric": float(best_metric),
            "history": history,
        }
        
        with open(os.path.join(config.OUT_DIR, "results3.json"), "w") as f:
            json.dump(results, f, indent=2)
        
        print(f"\nResults saved to {config.OUT_DIR}")
    
    return test_acc, test_f1



In [9]:

# ==============================================================================
# NOTEBOOK LAUNCHER
# ==============================================================================

def launch_from_notebook(num_processes: int = 6):
    """Launch distributed training from Jupyter notebook."""
    print(f"Launching training on {num_processes} GPUs...")
    return notebook_launcher(
        training_function,
        args=(),
        num_processes=num_processes,
        mixed_precision=config.MIXED_PRECISION,
        use_port=30001,   # change from default 29500
    )


def _in_notebook() -> bool:
    return 'ipykernel' in sys.modules



In [10]:

# ==============================================================================
# CLI ENTRY POINT
# ==============================================================================

def main():
    # call your actual training entrypoint directly
    training_function()

if __name__ == "__main__":
    main()
    
# if __name__ == "__main__":
#     training_function()
# Fix shared memory issue
# config.NUM_WORKERS = 0  # Prevents the bus error

# # Launch training on ALL 6 GPUs
# launch_from_notebook(num_processes=6)

Launching training on 6 GPUs...
Launching training on 6 CUDAs.


W1221 22:27:55.065000 36189 site-packages/torch/multiprocessing/spawn.py:169] Terminating process 36421 via signal SIGTERM
W1221 22:27:55.069000 36189 site-packages/torch/multiprocessing/spawn.py:169] Terminating process 36422 via signal SIGTERM
W1221 22:27:55.072000 36189 site-packages/torch/multiprocessing/spawn.py:169] Terminating process 36425 via signal SIGTERM
W1221 22:27:55.073000 36189 site-packages/torch/multiprocessing/spawn.py:169] Terminating process 36429 via signal SIGTERM
W1221 22:27:55.075000 36189 site-packages/torch/multiprocessing/spawn.py:169] Terminating process 36433 via signal SIGTERM
E1221 22:27:55.148000 36189 site-packages/torch/distributed/elastic/multiprocessing/api.py:737] failed (exitcode: 1) local_rank: 0 (pid: 36417) of fn: training_function (start_method: fork)
E1221 22:27:55.148000 36189 site-packages/torch/distributed/elastic/multiprocessing/api.py:737] Traceback (most recent call last):
E1221 22:27:55.148000 36189 site-packages/torch/distributed/elas

ChildFailedError: 
============================================================
training_function FAILED
------------------------------------------------------------
Failures:
  <NO_OTHER_FAILURES>
------------------------------------------------------------
Root Cause (first observed failure):
[0]:
  time      : 2025-12-21_22:27:54
  host      : jupyter-notebook-dev-1-674f44bc94-pr7ts
  rank      : 0 (local_rank: 0)
  exitcode  : 1 (pid: 36417)
  error_file: /tmp/torchelastic_x0s1r54s/none_3xqvcfmj/attempt_0/0/error.json
  traceback : Traceback (most recent call last):
    File "/opt/conda/lib/python3.12/site-packages/torch/distributed/elastic/multiprocessing/errors/__init__.py", line 355, in wrapper
      return f(*args, **kwargs)
             ^^^^^^^^^^^^^^^^^^
    File "/tmp/ipykernel_36189/2273625663.py", line 8, in training_function
      accelerator = Accelerator(
                    ^^^^^^^^^^^^
    File "/opt/conda/lib/python3.12/site-packages/accelerate/accelerator.py", line 